# FraudaZero — 02. Validação e experimentos

1. **Validação** (A4, pronta): estratégia, métricas e diagnóstico do baseline (regressão logística).
2. **Comparação de modelos** (A6) e **otimização** (A7): a fazer, usando as mesmas funções (seção 6).

Funções usadas: `src/validation.py` (holdout temporal, k-fold estratificado, teste no stream, gráficos) e `src/train.py` (`baseline_model`, `evaluate_and_log`). Todo modelo é um `Pipeline` **não treinado**: cada função faz `clone` e treina do zero, então escala e reamostragem são ajustadas só nos dados de treino de cada divisão.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("DATA_DIR", str(ROOT / "data"))
sys.path.insert(0, str(ROOT))

from src import validation as val
from src.train import baseline_model

FIGS = ROOT / "reports" / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
C_TRAIN, C_VALID = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

X, y = val.load_train()  # ordenado por Time, sem as 840 cópias exatas
print(X.shape, "| fraudes:", int(y.sum()), f"| prevalência: {y.mean():.4%}")

## 1. Métricas (definidas antes de qualquer comparação)

| Papel | Métrica | Por quê |
|---|---|---|
| **Principal** | **PR-AUC** (`average_precision_score`) | Resume precisão × recall da classe fraude em todos os limiares. A referência de um modelo aleatório é a prevalência (0,0018), então a escala é informativa. |
| Secundárias | Recall, precisão e F1 da classe fraude | Mostram o comportamento no limiar em uso (0,5 até a A8 escolher outro). |
| Referência | ROC-AUC | Comparação com a literatura. Fica otimista em bases desbalanceadas: o denominador da taxa de falsos positivos são ~227 mil legítimas, então milhares de falsos alarmes quase não a movem. |

**Acurácia não é usada:** com 0,18% de fraudes, responder "legítima" para tudo dá 99,82% de acurácia e recall zero.

## 2. Holdout temporal

O `train.csv` já está em ordem de `Time`. Treinamos nos primeiros 80% e validamos nos últimos 20%: é a situação real do sistema, que aprende com o passado e pontua transações futuras. Nenhuma linha do futuro entra no treino.

In [ ]:
hold = val.holdout_temporal(baseline_model(), X, y)
cut = int(len(X) * 0.8)
print(f"treino: {cut} linhas ({int(y.iloc[:cut].sum())} fraudes) | "
      f"validação: {len(X) - cut} linhas ({int(y.iloc[cut:].sum())} fraudes)")
print(f"Time do corte: {X.Time.iloc[cut]:.0f} s | tempo de treino: {hold['fit_seconds']:.1f} s")
tab = pd.DataFrame({"treino": hold["train"], "validação": hold["valid"]})
tab["diferença (treino − val.)"] = tab["treino"] - tab["validação"]
tab.round(4)

**Leitura.**
- **Sem overfitting:** a PR-AUC de treino (0,7255) e a de validação (0,7330) são praticamente iguais (diferença de −0,007). O mesmo vale para ROC-AUC (0,987 contra 0,974) e recall (0,919 contra 0,904).
- **Indício de underfitting:** o modelo não passa de ~0,73 de PR-AUC nem no próprio treino. Um modelo linear não captura relações como os picos de `Amount` (Fig. 2 da EDA). O ganho deve vir dos modelos não lineares da A6.
- **A precisão de treino (7,5%) é maior que a de validação (2,2%)** sem que isso seja overfitting: a validação tem menos fraudes (52 em 45.401, 0,11%) que o treino (347 em 181.604, 0,19%), e a precisão cai junto com a prevalência.

In [ ]:
fig = val.plot_pr_curve(hold["y_valid"], hold["scores_valid"],
                        title="Baseline — curva PR (holdout temporal)", name="Regressão logística")
fig.savefig(FIGS / "val_01_baseline_pr_holdout.png")
fig = val.plot_confusion(hold["y_valid"], hold["scores_valid"],
                         title="Baseline — holdout temporal")
fig.savefig(FIGS / "val_02_baseline_confusao_holdout.png")

**Leitura das figuras.**
- **Curva PR:** a precisão fica em 1,0 até cerca de 48% de recall (as fraudes mais óbvias recebem os maiores scores) e despenca depois de ~78% de recall. A PR-AUC de 0,73 resume essa curva.
- **Matriz de confusão (limiar 0,5):** 47 das 52 fraudes detectadas (recall de 90,4%), mas com **2.076 falsos alarmes** (precisão de 2,2%). Com `class_weight="balanced"`, o modelo empurra os scores para cima, e o limiar padrão de 0,5 deixa de ser adequado. A tabela abaixo mostra o efeito de mudar o limiar (escolha final na A8).

In [ ]:
from sklearn.metrics import precision_recall_curve

p, r, t = precision_recall_curve(hold["y_valid"], hold["scores_valid"])
rows = []
for target in [0.9, 0.8, 0.7]:
    i = np.where(r[:-1] >= target)[0][-1]  # maior limiar que ainda dá esse recall
    rows.append({"recall mínimo": target, "limiar": t[i], "recall": r[i], "precisão": p[i]})
pd.DataFrame(rows).style.format(precision=4).format({"limiar": "{:.6f}"})

**Leitura.** Exigir 80% de recall em vez de 90% sobe a precisão de 2,6% para 19,3%; aceitar 71% de recall leva a precisão a 67,3%. O limiar certo depende do custo de uma fraude perdida contra o custo de um falso alarme: é a decisão da A8 (valor de `FRAUD_THRESHOLD`). Os limiares ficam muito perto de 1 porque o `class_weight` desloca os scores; em outros modelos os valores serão diferentes.

## 3. Stratified k-fold (k = 5) e estabilidade

Cada fold mantém a proporção de fraudes (~80 fraudes por fold de validação). A semente é fixa (`SEED = 42` em `src/validation.py`), então **todos os modelos da A6 são avaliados exatamente nos mesmos folds**, o que permite comparar fold a fold.

In [ ]:
folds = val.cv_stratified(baseline_model(), X, y, k=5)
cols = ["fold", "fit_seconds", "train_pr_auc", "valid_pr_auc", "valid_roc_auc",
        "valid_recall", "valid_precision", "valid_f1"]
display(folds[cols].round(4))

summary = pd.DataFrame({
    "média": folds.drop(columns="fold").mean(),
    "desvio": folds.drop(columns="fold").std(),
})
summary["coef. variação"] = summary["desvio"] / summary["média"]
summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(folds.fold, folds.train_pr_auc, "o-", color=C_TRAIN, lw=2, ms=8, label="Treino do fold")
ax.plot(folds.fold, folds.valid_pr_auc, "o-", color=C_VALID, lw=2, ms=8, label="Validação do fold")
ax.axhline(hold["valid"]["pr_auc"], color="#52514e", ls="--", lw=1)
ax.annotate(f"holdout temporal = {hold['valid']['pr_auc']:.3f}",
            (4.6, hold["valid"]["pr_auc"] + 0.002), va="bottom", fontsize=9, color="#52514e")
ax.set_xticks(folds.fold)
ax.set_xlabel("Fold (stratified k-fold, k = 5)")
ax.set_ylabel("PR-AUC")
ax.set_xlim(0.7, 5.3)
ax.set_title("Baseline — estabilidade da PR-AUC entre folds")
ax.set_ylim(0.57, 0.78)
ax.legend(frameon=False, loc="lower center", ncol=2)
fig.savefig(FIGS / "val_03_baseline_estabilidade_folds.png")

**Leitura (estabilidade).**
- **PR-AUC de validação: 0,7148 ± 0,0657**, de 0,639 (fold 2) a 0,766 (fold 1); coeficiente de variação de 9,2%. A **ROC-AUC é bem mais estável (0,9708 ± 0,0090)**, e o recall também (0,8947 ± 0,0173).
- A PR-AUC oscila porque cada fold tem só ~80 fraudes: algumas fraudes com score baixo já mudam bastante a área. **Consequência para a A6:** diferenças de PR-AUC menores que ~0,07 entre modelos estão dentro do ruído; compare os modelos **fold a fold** (os folds são os mesmos) e olhe também o holdout temporal.
- **Treino × validação:** as médias são iguais (0,7147 × 0,7148); só a variação muda (desvio de 0,016 no treino contra 0,066 na validação). Não há overfitting: o desvio maior vem do tamanho pequeno da validação, não de memorização.

## 4. Holdout temporal × k-fold estratificado

| Estratégia | PR-AUC | O que mede |
|---|---|---|
| Holdout temporal (80/20) | 0,733 | Desempenho no "futuro": treina no passado, valida depois. É o cenário real de produção. |
| Stratified 5-fold | 0,715 ± 0,066 | Média e **variabilidade**, usando todos os dados para validar uma vez. Ignora a ordem do tempo. |

O k-fold embaralha as linhas, então o modelo pode treinar com transações posteriores às da validação. Neste baseline isso não inflou o resultado: o holdout (0,733) está dentro do intervalo dos folds e perto da média. Para modelos mais flexíveis (KNN, árvores), o k-fold pode ficar otimista, porque transações próximas no tempo tendem a se parecer. Por isso a A6 deve **reportar as duas estimativas** e a escolha final deve olhar principalmente o holdout temporal. O `stream.csv` (últimos 20% da base) fica reservado para **uma única avaliação final** do campeão (`val.evaluate_on_stream`, A8).

## 5. Leave-One-Out: por que não se aplica

O LOO treina um modelo para cada linha e valida nela. Rodamos numa subamostra pequena só para medir o custo.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict

# Subamostra pequena só para demonstrar o custo: 50 fraudes + 950 legítimas
rng = np.random.RandomState(42)
idx = np.r_[rng.choice(np.where(y == 1)[0], 50, replace=False),
            rng.choice(np.where(y == 0)[0], 950, replace=False)]
Xs, ys = X.iloc[idx], y.iloc[idx]

start = time.perf_counter()
loo_scores = cross_val_predict(baseline_model(), Xs, ys, cv=LeaveOneOut(),
                               method="predict_proba", n_jobs=-1)[:, 1]
loo_s = time.perf_counter() - start
print(f"LOO em n = {len(Xs)}: {len(Xs)} treinos, {loo_s:.1f} s no total "
      f"(paralelo em {os.cpu_count()} núcleos)")
print("PR-AUC das previsões agregadas do LOO:", round(val.metrics(ys, loo_scores)["pr_auc"], 4))

full_fit = np.mean(folds.fit_seconds)  # um treino em ~80% da base
n = len(X)
print(f"\nProjeção para a base inteira (n = {n}): {n} treinos x {full_fit:.2f} s "
      f"= {n * full_fit / 3600:.1f} horas só para a regressão logística, em 1 núcleo")
print(f"Com 5-fold: 5 treinos x {full_fit:.2f} s = {5 * full_fit:.1f} s")

**Leitura e justificativa.**
1. **Custo:** o LOO exige um treino por linha. Na base inteira seriam **227.005 treinos**; com ~0,3 s por treino (medido acima nos folds), só a regressão logística, o modelo mais barato, levaria **cerca de 20 horas** num núcleo, contra ~2 s do 5-fold. Random Forest, AdaBoost e MLP levam de dezenas a centenas de vezes mais por treino: semanas ou meses. (Os tempos variam um pouco entre execuções; a ordem de grandeza, não.)
2. **Métricas por fold não existem:** cada validação tem **uma única linha**. PR-AUC, recall e precisão não são definidos para uma linha, então não há análise de estabilidade por fold, que é justamente o que a especificação pede. Só dá para agregar as previsões no fim.
3. **Variância alta:** os 227 mil conjuntos de treino são quase idênticos entre si, e a estimativa final tende a ter variância alta.
4. **Ignora o tempo**, como o k-fold, sem a vantagem do custo baixo.

A PR-AUC de 0,875 na subamostra **não é comparável** com as outras: a subamostra tem 5% de fraudes (50 em 1.000), contra 0,18% na base, e a PR-AUC sobe com a prevalência. Isso reforça que toda avaliação deve usar a proporção real de fraudes.

**Conclusão:** com ~285 mil linhas, o LOO é inviável computacionalmente e inadequado para métricas da classe rara. Usamos holdout temporal + stratified 5-fold.

## 6. Comparação de modelos (A6) e otimização (A7) — a fazer

Modelo de uso (substitua o classificador; o resto é igual para todos):

```python
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.neighbors import KNeighborsClassifier

from src.features import preprocessor
from src.train import evaluate_and_log

# KNN: undersampling DENTRO do Pipeline (só no treino de cada fold), mantendo todas as fraudes.
knn = ImbPipeline([
    ("prep", preprocessor()),
    ("under", RandomUnderSampler(sampling_strategy=0.02, random_state=42)),  # ~1 fraude : 50 legítimas
    ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1)),
])
evaluate_and_log(knn, run_name="knn", params={"model": "KNN", "k": 15, "under": 0.02},
                 experiment="comparacao")
```

- Para modelos rápidos, a `evaluate_and_log` faz holdout + 5-fold + treino final e registra tudo no MLflow (métricas, `cv_folds.csv`, matriz de confusão, curva PR, modelo). Para explorar antes de registrar, use direto `val.holdout_temporal(...)` e `val.cv_stratified(...)`.
- **Não reamostre fora do Pipeline** e **não avalie no `stream.csv`** até a escolha final (A8).